# 📚 Online Bookstore Database Management System
### MySQL Database Project using Jupyter Notebook

**Project Domain:** Bookstore Management & Sales Analytics  
**Database:** MySQL 8.0+  
**Notebook Purpose:** Build, manage, automate, and analyze an online bookstore database using SQL and Python.

---

## 1. Abstract

The Online Bookstore Database Management System is designed to manage bookstore operations through a relational database. It stores book details, customer information, orders, order items, and administrator records.

The project applies SQL concepts such as primary keys, foreign keys, constraints, indexes, triggers, stored procedures, functions, views, and transactions. It automates order processing, inventory validation, subtotal calculation, and order-total calculation. Analytical queries are used to examine sales, customer spending, inventory, and order status.

This notebook demonstrates the database implementation and analysis step-by-step.

## 2. Introduction

Bookstores need to maintain accurate records of books, customers, inventory, and transactions. Manual record keeping can lead to duplicate information, incorrect billing, stock errors, and difficulty preparing reports.

This project uses MySQL to create a centralized database and Python in Jupyter Notebook to connect to the database, execute SQL commands, and display query results.

## 3. Problem Statement

Manual bookstore management can make it difficult to:

- Maintain accurate book inventory.
- Track customer orders and purchases.
- Calculate order bills accurately.
- Prevent orders when stock is insufficient.
- Identify popular books and profitable categories.
- Generate sales and customer reports efficiently.

The project aims to address these issues using a relational database and automated SQL operations.

## 4. Objectives

- Design a relational database for bookstore operations.
- Maintain book, customer, order, order-item, and administrator records.
- Enforce data integrity using keys and constraints.
- Validate stock availability during order processing.
- Automatically calculate subtotals and order totals.
- Implement triggers, stored procedures, and functions.
- Demonstrate transactions and indexing.
- Generate analytical reports using SQL queries.

## 5. Scope

### Included
- Book information management
- Customer records
- Order and order-item management
- Inventory tracking
- Stock validation and reduction
- Automatic subtotal and total calculation
- Order status tracking
- Sales and customer analytics

### Future Enhancements
- Payment integration
- Shopping cart
- Customer login
- Delivery tracking
- Book reviews and ratings
- Power BI dashboard
- Stock restoration for cancelled orders

## 6. Tools & Technologies

| Tool / Technology | Purpose |
|---|---|
| MySQL 8.0+ | Relational database |
| Jupyter Notebook | Interactive project development |
| Python | Connecting to MySQL and displaying results |
| SQL | Database creation, operations, and analysis |
| mysql-connector-python | Python-to-MySQL connection |
| Pandas | Displaying query results as DataFrames |

### Install the required Python packages

Run this cell if the packages are not already installed.

In [ ]:
%pip install mysql-connector-python pandas

## 7. Database Connection

> **Before running:** Ensure MySQL Server is installed and running. Enter your own MySQL credentials below. Do not upload real passwords to GitHub.

In [ ]:
import mysql.connector
import pandas as pd
from mysql.connector import Error

# Update these settings for your local MySQL installation
MYSQL_HOST = "localhost"
MYSQL_USER = "root"
MYSQL_PASSWORD = "YOUR_MYSQL_PASSWORD"

# Connect to MySQL server (without selecting a database initially)
connection = mysql.connector.connect(
    host=MYSQL_HOST,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD
)

cursor = connection.cursor()

print("Connected to MySQL successfully!")

## 8. Create the Database

The database is named `Bookstore`.

> ⚠️ The `DROP DATABASE` command deletes the existing Bookstore database and its data. Run this reset cell only when you intentionally want to recreate the project from scratch.

In [ ]:
cursor.execute("DROP DATABASE IF EXISTS Bookstore")
cursor.execute("CREATE DATABASE Bookstore")
cursor.execute("USE Bookstore")

connection.database = "Bookstore"

print("Bookstore database created and selected.")

## 9. Create Tables

The project contains five tables:

1. `Books`
2. `Customers`
3. `Orders`
4. `Order_Items`
5. `Admins`

Primary keys uniquely identify records. Foreign keys connect related tables. Constraints help validate data.

In [ ]:
create_tables = [
    '''
    CREATE TABLE Books (
        book_id INT AUTO_INCREMENT PRIMARY KEY,
        title VARCHAR(255) NOT NULL,
        author VARCHAR(255) NOT NULL,
        price DECIMAL(10,2) NOT NULL CHECK (price >= 0),
        stock INT NOT NULL DEFAULT 0 CHECK (stock >= 0),
        category VARCHAR(100),
        published_date DATE
    )
    ''',
    '''
    CREATE TABLE Customers (
        customer_id INT AUTO_INCREMENT PRIMARY KEY,
        name VARCHAR(255) NOT NULL,
        email VARCHAR(255) UNIQUE NOT NULL,
        phone VARCHAR(20),
        address TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    ''',
    '''
    CREATE TABLE Orders (
        order_id INT AUTO_INCREMENT PRIMARY KEY,
        customer_id INT NOT NULL,
        order_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        total_amount DECIMAL(10,2) NOT NULL DEFAULT 0 CHECK (total_amount >= 0),
        status ENUM('Pending', 'Completed', 'Cancelled') DEFAULT 'Pending',
        CONSTRAINT fk_orders_customer
            FOREIGN KEY (customer_id) REFERENCES Customers(customer_id)
            ON DELETE RESTRICT
    )
    ''',
    '''
    CREATE TABLE Order_Items (
        order_item_id INT AUTO_INCREMENT PRIMARY KEY,
        order_id INT NOT NULL,
        book_id INT NOT NULL,
        quantity INT NOT NULL CHECK (quantity > 0),
        unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price >= 0),
        subtotal DECIMAL(10,2) NOT NULL CHECK (subtotal >= 0),
        CONSTRAINT fk_order_items_order
            FOREIGN KEY (order_id) REFERENCES Orders(order_id)
            ON DELETE CASCADE,
        CONSTRAINT fk_order_items_book
            FOREIGN KEY (book_id) REFERENCES Books(book_id)
            ON DELETE RESTRICT
    )
    ''',
    '''
    CREATE TABLE Admins (
        admin_id INT AUTO_INCREMENT PRIMARY KEY,
        username VARCHAR(100) NOT NULL UNIQUE,
        password_hash VARCHAR(255) NOT NULL,
        role ENUM('Manager', 'Staff') DEFAULT 'Staff'
    )
    '''
]

for statement in create_tables:
    cursor.execute(statement)

connection.commit()
print("All five tables created successfully.")

### 9.1 Verify Table Structure

`SHOW TABLES` displays the tables created inside the selected database.

In [ ]:
cursor.execute("SHOW TABLES")
pd.DataFrame(cursor.fetchall(), columns=["Tables"])

## 10. Insert Sample Data

The following sample records are used to test the database and demonstrate analytical queries.

In [ ]:
books_data = [
    ('The Great Gatsby', 'F. Scott Fitzgerald', 10.99, 50, 'Fiction', '1925-04-10'),
    ('1984', 'George Orwell', 8.99, 40, 'Dystopian', '1949-06-08'),
    ('To Kill a Mockingbird', 'Harper Lee', 12.50, 30, 'Classic', '1960-07-11'),
    ('The Catcher in the Rye', 'J. D. Salinger', 9.99, 20, 'Novel', '1951-07-16'),
    ('Moby Dick', 'Herman Melville', 15.75, 25, 'Adventure', '1851-10-18')
]

cursor.executemany(
    '''
    INSERT INTO Books (title, author, price, stock, category, published_date)
    VALUES (%s, %s, %s, %s, %s, %s)
    ''',
    books_data
)

customers_data = [
    ('Alice Johnson', 'alice@example.com', '1234567890', '123 Main Street'),
    ('Bob Smith', 'bob@example.com', '0987654321', '456 Elm Street'),
    ('Charlie Brown', 'charlie@example.com', '1122334455', '789 Pine Street'),
    ('David Wilson', 'david@example.com', '2233445566', '101 Oak Street'),
    ('Eve Adams', 'eve@example.com', '3344556677', '202 Birch Street')
]

cursor.executemany(
    '''
    INSERT INTO Customers (name, email, phone, address)
    VALUES (%s, %s, %s, %s)
    ''',
    customers_data
)

admins_data = [
    ('admin1', 'REPLACE_WITH_SECURE_HASH', 'Manager'),
    ('staff1', 'REPLACE_WITH_SECURE_HASH', 'Staff'),
    ('admin2', 'REPLACE_WITH_SECURE_HASH', 'Manager'),
    ('staff2', 'REPLACE_WITH_SECURE_HASH', 'Staff'),
    ('staff3', 'REPLACE_WITH_SECURE_HASH', 'Staff')
]

cursor.executemany(
    '''
    INSERT INTO Admins (username, password_hash, role)
    VALUES (%s, %s, %s)
    ''',
    admins_data
)

connection.commit()
print("Sample books, customers, and admin records inserted.")

## 11. Constraints

Constraints protect the database from invalid data.

| Constraint | Example in this project |
|---|---|
| PRIMARY KEY | Identifies each book/customer/order |
| FOREIGN KEY | Connects orders with customers |
| NOT NULL | Book title cannot be empty |
| UNIQUE | Customer email must be unique |
| CHECK | Price and stock cannot be negative |
| DEFAULT | New orders default to Pending |
| ENUM | Order status is restricted to allowed values |

### View sample books

In [ ]:
cursor.execute("SELECT * FROM Books ORDER BY title")
pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])

## 12. Indexes

Indexes can improve retrieval speed for frequently searched or joined columns.

The project creates indexes on customer email, order customer ID, order-item order ID, order-item book ID, and book category.

> Note: A UNIQUE constraint on email already creates an index in MySQL. The explicit email index in the original script is redundant and is included here to demonstrate index creation; in a production design, avoid keeping redundant indexes.

In [ ]:
index_statements = [
    "CREATE INDEX idx_orders_customer_id ON Orders(customer_id)",
    "CREATE INDEX idx_order_items_order_id ON Order_Items(order_id)",
    "CREATE INDEX idx_order_items_book_id ON Order_Items(book_id)",
    "CREATE INDEX idx_books_category ON Books(category)"
]

for statement in index_statements:
    cursor.execute(statement)

connection.commit()
print("Indexes created.")

## 13. Triggers

Triggers automatically execute when specified database events occur.

### Trigger 1: Validate Stock and Reduce Inventory

Before inserting an order item:
- Check that quantity is positive.
- Check that the book exists.
- Check whether stock is sufficient.
- Reduce stock by the ordered quantity.

In [ ]:
trigger_stock = '''
CREATE TRIGGER trg_order_items_before_insert
BEFORE INSERT ON Order_Items
FOR EACH ROW
BEGIN
    DECLARE current_stock INT DEFAULT NULL;

    IF NEW.quantity <= 0 THEN
        SIGNAL SQLSTATE '45000'
            SET MESSAGE_TEXT = 'Quantity must be greater than zero';
    END IF;

    SELECT stock INTO current_stock
    FROM Books
    WHERE book_id = NEW.book_id
    FOR UPDATE;

    IF current_stock IS NULL THEN
        SIGNAL SQLSTATE '45000'
            SET MESSAGE_TEXT = 'Book does not exist';
    ELSEIF current_stock < NEW.quantity THEN
        SIGNAL SQLSTATE '45000'
            SET MESSAGE_TEXT = 'Insufficient stock';
    END IF;

    UPDATE Books
    SET stock = stock - NEW.quantity
    WHERE book_id = NEW.book_id;
END
'''

cursor.execute(trigger_stock)
connection.commit()
print("Stock validation trigger created.")

### Trigger 2: Calculate Item Subtotal

Formula:

**Subtotal = Unit Price × Quantity**

The subtotal is calculated automatically before an order-item row is inserted.

In [ ]:
trigger_subtotal = '''
CREATE TRIGGER trg_order_items_before_insert_price
BEFORE INSERT ON Order_Items
FOR EACH ROW
BEGIN
    SET NEW.subtotal = NEW.unit_price * NEW.quantity;
END
'''

cursor.execute(trigger_subtotal)
connection.commit()
print("Subtotal trigger created.")

### Trigger 3: Update Order Total

After an order item is inserted, the trigger recalculates the order total by summing all subtotals belonging to that order.

In [ ]:
trigger_total = '''
CREATE TRIGGER trg_order_items_after_insert
AFTER INSERT ON Order_Items
FOR EACH ROW
BEGIN
    UPDATE Orders
    SET total_amount = (
        SELECT COALESCE(SUM(subtotal), 0)
        FROM Order_Items
        WHERE order_id = NEW.order_id
    )
    WHERE order_id = NEW.order_id;
END
'''

cursor.execute(trigger_total)
connection.commit()
print("Order total trigger created.")

## 14. Stored Procedure: ProcessOrder

The stored procedure processes an order for one book.

### Parameters
- Customer ID
- Book ID
- Quantity

### Workflow
1. Validate quantity.
2. Start transaction.
3. Retrieve book price.
4. Create order.
5. Get generated order ID.
6. Insert order item.
7. Triggers validate stock, reduce stock, calculate subtotal, and update total.
8. Commit the transaction.
9. Roll back if an exception occurs.

In [ ]:
procedure_sql = '''
CREATE PROCEDURE ProcessOrder(
    IN p_customer_id INT,
    IN p_book_id INT,
    IN p_quantity INT
)
BEGIN
    DECLARE v_price DECIMAL(10,2) DEFAULT NULL;
    DECLARE v_order_id INT;

    DECLARE EXIT HANDLER FOR SQLEXCEPTION
    BEGIN
        ROLLBACK;
        RESIGNAL;
    END;

    IF p_quantity <= 0 THEN
        SIGNAL SQLSTATE '45000'
            SET MESSAGE_TEXT = 'Quantity must be greater than zero';
    END IF;

    START TRANSACTION;

    SELECT price INTO v_price
    FROM Books
    WHERE book_id = p_book_id;

    IF v_price IS NULL THEN
        SIGNAL SQLSTATE '45000'
            SET MESSAGE_TEXT = 'Book does not exist';
    END IF;

    INSERT INTO Orders (customer_id, total_amount, status)
    VALUES (p_customer_id, 0, 'Pending');

    SET v_order_id = LAST_INSERT_ID();

    INSERT INTO Order_Items (order_id, book_id, quantity, unit_price, subtotal)
    VALUES (v_order_id, p_book_id, p_quantity, v_price, v_price * p_quantity);

    COMMIT;

    SELECT v_order_id AS created_order_id,
           v_price * p_quantity AS order_total;
END
'''

cursor.execute(procedure_sql)
connection.commit()
print("ProcessOrder procedure created.")

### Test the Stored Procedure

The following call places an order for two copies of book ID 2 (1984) for customer ID 1.

In [ ]:
cursor.callproc("ProcessOrder", [1, 2, 2])

# Display result sets returned by the procedure
for result in cursor.stored_results():
    display(pd.DataFrame(result.fetchall(), columns=[col[0] for col in result.description]))

connection.commit()

## 15. User-Defined Function: CalculateDiscount

The function returns a 10% discount amount.

**Formula:** Discount = Amount × 0.10

Example: If the amount is 1,000, the discount is 100.

In [ ]:
function_sql = '''
CREATE FUNCTION CalculateDiscount(p_amount DECIMAL(10,2))
RETURNS DECIMAL(10,2)
DETERMINISTIC
NO SQL
BEGIN
    RETURN ROUND(p_amount * 0.10, 2);
END
'''

cursor.execute(function_sql)
connection.commit()
print("CalculateDiscount function created.")

### Test the Function

In [ ]:
cursor.execute("SELECT CalculateDiscount(1000.00) AS discount_amount")
pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])

## 16. Create Sample Orders

The stored procedure is called to create the remaining sample orders.

> This notebook's order IDs are generated automatically. Avoid assuming they will always be 1–5 if you rerun only some cells or have existing records.

In [ ]:
sample_order_calls = [
    (2, 5, 1),  # Bob: one Moby Dick
    (3, 4, 1),  # Charlie: one Catcher in the Rye
    (4, 1, 2),  # David: two Great Gatsby
    (5, 3, 1)   # Eve: one To Kill a Mockingbird
]

for args in sample_order_calls:
    cursor.callproc("ProcessOrder", args)
    # Drain procedure result sets
    for result in cursor.stored_results():
        result.fetchall()

connection.commit()
print("Additional sample orders processed.")

### Update Sample Order Statuses

For demonstration purposes, mark some orders as completed and one as cancelled.

Because the order IDs are generated dynamically, update based on the intended customer/book sample rows instead of hard-coded order IDs.

In [ ]:
# Mark the order for Alice as Completed
cursor.execute(
    '''
    UPDATE Orders o
    JOIN Order_Items oi ON oi.order_id = o.order_id
    JOIN Books b ON b.book_id = oi.book_id
    SET o.status = 'Completed'
    WHERE o.customer_id = 1 AND b.book_id = 2
    '''
)

# Mark David's sample order as Completed
cursor.execute(
    '''
    UPDATE Orders o
    JOIN Order_Items oi ON oi.order_id = o.order_id
    SET o.status = 'Completed'
    WHERE o.customer_id = 4 AND oi.book_id = 1
    '''
)

# Mark Charlie's sample order as Cancelled
cursor.execute(
    '''
    UPDATE Orders o
    JOIN Order_Items oi ON oi.order_id = o.order_id
    SET o.status = 'Cancelled'
    WHERE o.customer_id = 3 AND oi.book_id = 4
    '''
)

connection.commit()
print("Sample order statuses updated.")

## 17. Analytical Queries

Analytical queries help convert transaction data into useful business information.

### A. View All Books

In [ ]:
query = "SELECT * FROM Books ORDER BY title"
cursor.execute(query)
pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])

### B. Low Stock Analysis

Books with stock quantity less than or equal to 10.

In [ ]:
query = '''
SELECT book_id, title, stock
FROM Books
WHERE stock <= 10
ORDER BY stock ASC
'''
cursor.execute(query)
pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])

### C. Customer Order Details

In [ ]:
query = '''
SELECT
    c.name AS customer_name,
    o.order_id,
    o.order_date,
    o.total_amount,
    o.status
FROM Customers c
JOIN Orders o ON o.customer_id = c.customer_id
ORDER BY o.order_date DESC
'''
cursor.execute(query)
pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])

### D. Best-Selling Books

Cancelled orders are excluded from this calculation.

In [ ]:
query = '''
SELECT
    b.book_id,
    b.title,
    b.category,
    SUM(oi.quantity) AS total_sold
FROM Books b
JOIN Order_Items oi ON oi.book_id = b.book_id
JOIN Orders o ON o.order_id = oi.order_id
WHERE o.status <> 'Cancelled'
GROUP BY b.book_id, b.title, b.category
ORDER BY total_sold DESC, b.title
'''
cursor.execute(query)
pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])

### E. Sales by Category

Calculates the quantity sold and revenue generated by each category, excluding cancelled orders.

In [ ]:
query = '''
SELECT
    b.category,
    SUM(oi.quantity) AS books_sold,
    SUM(oi.subtotal) AS revenue
FROM Books b
JOIN Order_Items oi ON oi.book_id = b.book_id
JOIN Orders o ON o.order_id = oi.order_id
WHERE o.status <> 'Cancelled'
GROUP BY b.category
ORDER BY revenue DESC
'''
cursor.execute(query)
pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])

### F. Revenue by Order Status

In [ ]:
query = '''
SELECT
    status,
    COUNT(*) AS order_count,
    SUM(total_amount) AS total_amount
FROM Orders
GROUP BY status
'''
cursor.execute(query)
pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])

### G. Customer Spending Summary

Cancelled order amounts are excluded from total spending.

In [ ]:
query = '''
SELECT
    c.customer_id,
    c.name,
    COUNT(DISTINCT o.order_id) AS number_of_orders,
    COALESCE(
        SUM(CASE
            WHEN o.status <> 'Cancelled' THEN o.total_amount
            ELSE 0
        END), 0
    ) AS total_spent
FROM Customers c
LEFT JOIN Orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_spent DESC
'''
cursor.execute(query)
pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])

### H. Order Item Details

In [ ]:
query = '''
SELECT
    b.title,
    oi.order_id,
    oi.quantity,
    oi.unit_price,
    oi.subtotal
FROM Books b
JOIN Order_Items oi ON oi.book_id = b.book_id
ORDER BY b.title, oi.order_id
'''
cursor.execute(query)
pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])

## 18. Create a View: OrderSummary

A view is a virtual table created from a SQL query. This view combines order details with customer names.

In [ ]:
view_sql = '''
CREATE VIEW OrderSummary AS
SELECT
    o.order_id,
    c.name AS customer_name,
    o.order_date,
    o.total_amount,
    o.status
FROM Orders o
JOIN Customers c ON c.customer_id = o.customer_id
'''

cursor.execute(view_sql)
connection.commit()

cursor.execute("SELECT * FROM OrderSummary")
pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])

## 19. Transaction Demonstration

Transactions group database changes. `COMMIT` saves the changes, while `ROLLBACK` undoes uncommitted changes.

The next example demonstrates a stock update with validation. This is a separate demonstration from order processing.

In [ ]:
# Demonstrate a transaction by decrementing stock only if stock is available.
connection.start_transaction()

cursor.execute(
    "UPDATE Books SET stock = stock - 1 WHERE book_id = 1 AND stock >= 1"
)

if cursor.rowcount == 1:
    connection.commit()
    print("Stock update committed successfully.")
else:
    connection.rollback()
    print("Stock was unavailable. Transaction rolled back.")

## 20. Query Optimization with EXPLAIN

`EXPLAIN` displays how MySQL plans to execute a query. It helps investigate whether indexes are used and how rows may be accessed.

In [ ]:
cursor.execute("EXPLAIN SELECT * FROM Orders WHERE customer_id = 5")
pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])

## 21. Business Insights

The SQL queries can help management understand:

- **Best-selling books:** Identify books with the highest purchase quantities.
- **Category performance:** Compare revenue across book categories.
- **Customer spending:** Identify customers with higher purchase values.
- **Inventory risk:** Identify books with low stock.
- **Order status:** Monitor pending, completed, and cancelled orders.
- **Operational efficiency:** Automated triggers reduce manual calculation and stock updates.

Insights should be interpreted from the results produced by the notebook, rather than assumed in advance.

## 22. Testing & Validation

| Test | Expected Result |
|---|---|
| Insert valid book | Record inserted |
| Negative price | Rejected by CHECK constraint |
| Negative stock | Rejected by CHECK constraint |
| Duplicate customer email | Rejected by UNIQUE constraint |
| Order quantity greater than stock | Trigger raises insufficient-stock error |
| Quantity zero or negative | Rejected |
| Insert order item | Subtotal calculated |
| Insert order item | Order total recalculated |
| Delete an order | Related items deleted through CASCADE |
| Delete referenced book | Restricted |
| Discount function test | Returns 10% of amount |
| Commit transaction | Changes saved |
| Rollback transaction | Uncommitted changes undone |

## 23. Limitations

- This is a database backend prototype and does not include a graphical interface.
- The stored procedure currently processes one book per call.
- The dataset is small and created for demonstration.
- Payment, shipping, and customer authentication are not implemented.
- Cancelled orders do not automatically restore inventory in the current design.
- Admin password strings are placeholders and are not actual secure password hashes.
- The email index is redundant with the UNIQUE constraint.
- The separate transaction example can change stock if committed; run it only when you intend to test that behavior.

## 24. Future Enhancements

- Build a web interface using Flask or Django.
- Add a shopping cart and customer login.
- Integrate payment and delivery services.
- Restore stock automatically when an order is cancelled.
- Support multiple books in one order-processing procedure.
- Create Power BI dashboards for sales and inventory.
- Add monthly and yearly sales reports.
- Add book reviews, ratings, and recommendations.
- Improve role-based access control and secure password storage.
- Add automatic low-stock notifications.

## 25. Conclusion

The Online Bookstore Database Management System demonstrates practical implementation of relational database concepts using MySQL. It organizes books, customers, orders, order items, and administrators into related tables.

The project applies constraints for data integrity, triggers for stock and billing automation, stored procedures for order processing, functions for discount calculations, views for reporting, indexes for retrieval support, and transactions for grouped operations.

Analytical queries help extract information about sales, customers, inventory, and order status. This project provides practical experience in SQL, database design, automation, transaction management, and business analysis, and can be extended into a full-stack bookstore application.

## 26. References

- MySQL 8.0 Documentation: https://dev.mysql.com/doc/
- MySQL Workbench Documentation: https://dev.mysql.com/doc/workbench/en/
- SQL and Relational Database Design learning resources

---

## Notebook Cleanup

When finished, close the cursor and connection. Do not run this until you have completed all queries.

In [ ]:
cursor.close()
connection.close()
print("MySQL connection closed.")